# 5. Topic Modeling & Advanced Visualization

Notebook ini menjalankan Topic Modeling (LDA) untuk menemukan tema-tema tersembunyi dalam ulasan, dan men-generate seluruh chart dan grafik untuk presentasi akhir.

In [1]:
import sys
from pathlib import Path
import pandas as pd

base_dir = Path.cwd().parent
sys.path.insert(0, str(base_dir))

from config.settings import DATA_PROCESSED, DATA_RESULTS
from analysis.topic_modeler import TopicModeler
from visualization.charts import ChartGenerator
from visualization.wordcloud_gen import WordCloudGenerator
from visualization.dashboard import DashboardGenerator

## 5.1 Topic Modeling (Opsional, tapi sangat insightfull)

In [2]:
absa_path = DATA_PROCESSED / "absa_data.csv"
if absa_path.exists():
    df_final = pd.read_csv(absa_path)
    
    print("Menjalankan Topic Modeling (LDA)...")
    modeler = TopicModeler()
    # Fokus pada review negatif agar tahu keluhan spesifik apa saja selain aspect yg kita tentukan
    neg_texts = df_final[df_final['consensus_label'] == 'negative']['clean_text_no_stop']
    
    if not neg_texts.empty and len(neg_texts) > 10:
        topics_df = modeler.fit_lda(neg_texts, n_topics=5)
        print("\n=== Topik Keluhan yang Ditemukan ===")
        display(topics_df)
    else:
        print("Data ulasan negatif terlalu sedikit untuk Topic Modeling.")
else:
    print("Data absa belum siap.")

Menjalankan Topic Modeling (LDA)...

=== Topik Keluhan yang Ditemukan ===


,topic_id,top_words,weight_sum
0,0,"protein, shake, protein shake, whey, bitter, w...",89.62
1,1,"yogurt, naked, greek, greek yogurt, 10g, 7g, 5...",37.15
2,2,"protein, whey, and, isolate, powder, chocolate...",316.00
3,3,"protein, and, taste, shake, whey, its, no, som...",403.41
4,4,"protein, shake, protein shake, enak, tidak, ma...",737.42


## 5.2 Generate Visualizations

Akan men-generate PNG high-res (300 DPI) di folder `outputs/figures/`.

In [3]:
if 'df_final' in locals():
    print("Mulai generate grafik & wordclouds...")
    
    # Load matrix dan hasil analisa lain
    try:
        aspect_matrix = pd.read_csv(DATA_RESULTS / "aspect_matrix.csv")
        pain_df = pd.read_csv(DATA_RESULTS / "pain_hierarchy.csv")
        complaint_kw = pd.read_csv(DATA_RESULTS / "complaint_keywords.csv")
    except Exception as e:
        print(f"File hasil analisis belum lengkap: {e}")
        aspect_matrix, pain_df, complaint_kw = None, None, None

    # 1. Charts
    charts = ChartGenerator()
    charts.generate_all(df_final, aspect_matrix, pain_df, complaint_kw)
    
    # 2. Word Clouds
    wclouds = WordCloudGenerator()
    wclouds.generate_all(df_final)
    
    # 3. Composite Dashboard
    dash = DashboardGenerator()
    dash.generate(df_final, aspect_matrix, pain_df)
    
    print("\nSelesai! Semua visualisasi telah tersimpan di folder 'outputs/figures/'.")

Mulai generate grafik & wordclouds...


/home/azril/Personal/Projects/DSAI/NUSFTC/nlp_social_listening/visualization/dashboard.py:72: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) DejaVu Sans.
  fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight", facecolor=BG)
/home/azril/Personal/Projects/DSAI/NUSFTC/nlp_social_listening/visualization/dashboard.py:72: UserWarning: Glyph 128225 (\N{SATELLITE ANTENNA}) missing from font(s) DejaVu Sans.
  fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight", facecolor=BG)
/home/azril/Personal/Projects/DSAI/NUSFTC/nlp_social_listening/visualization/dashboard.py:72: UserWarning: Glyph 128200 (\N{CHART WITH UPWARDS TREND}) missing from font(s) DejaVu Sans.
  fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight", facecolor=BG)
/home/azril/Personal/Projects/DSAI/NUSFTC/nlp_social_listening/visualization/dashboard.py:72: UserWarning: Glyph 128308 (\N{LARGE RED CIRCLE}) missing from font(s) DejaVu Sans.
  fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight", facecolor=BG)
/home/


Selesai! Semua visualisasi telah tersimpan di folder 'outputs/figures/'.
